## Identify missing oligos and plot them per label
### Outline:
1. Find number of oligos and their names `/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/experiment/results/assignment/assignIGVFDesign/reference.fa`
2. Find all assigned oligos with the barcodes
3. Take the difference
4. How many missings per group?

In [43]:
import pandas as pd
import matplotlib.pyplot as plt
import gzip

In [2]:
! pwd

/data/gpfs-1/work/users/kisa11_c/coding/80K_analysis/01_missing_sequences


In [ ]:
def make_header_list(fasta_file):
    header_list = []
    with open(fasta_file, 'r') as fh:
        for line in fh:
            if line.startswith('>'):
                header_list.append(line.strip().lstrip('>'))
    # make df from list
    header_df = pd.DataFrame(header_list)
    return header_df # or use cat reference.fa | grep ">" | awk '{print substr($0,2)}' > all_headers.tsv

In [2]:
all_headers = '/fast/work/groups/ag_kircher/MPRA/IGVF_Y1_design/experiment/results/assignment/assignIGVFDesign/all_ref_sequences.tsv'
all_headers = '/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/experiment/standard_results/results/assignment/standardAssignIGVFDesignNoTemp/reference/all_headers.tsv'
# read in all sequences as tsv
all_seq_df = pd.read_csv(all_headers, header=None, sep='\t')
all_seq_df.columns = ['oligo_name']
# print(all_seq_df.shape) # 80214
all_seq_df

,oligo_name
0,cardiac_neuro_cava_random:SKI|ENSG00000157933....
1,cardiac_neuro_cava_random:SKI|ENSG00000157933....
2,cardiac_neuro_cava_random:SKI|ENSG00000157933....
3,cardiac_neuro_cava_random:SKI|ENSG00000157933....
4,cardiac_neuro_cava_random:SKI|ENSG00000157933....
...,...
80210,MK:tile_2240|chr1-116244322+116244591|scramble...
80211,MK:tile_6675|chr11-2374617+2374886|scramble_ne...
80212,MK:tile_18415|chr17-71181691+71181960|scramble...
80213,MK:tile_14356|chr15-67031618+67031887|scramble...


In [3]:
# add second column which is split at first : and contains the label
all_seq_df['label'] = all_seq_df['oligo_name'].str.split(':').str[0]


In [5]:
all_seq_df
# show all unique labels
print(all_seq_df['label'].unique())
# print(len(all_seq_df['label'].unique())) # 29

['cardiac_neuro_cava_random' 'GC_Atrial_fib' 'GC_Liang' 'GC_Selvarajan'
 'GC_Mohlke' 'GC_Kircher' 'GC_Mendelian_variants' 'C_positive_heart_CAD'
 'GC_Cort_Chengyu' 'GC_GABA_Chengyu' 'GC_Glut_Chengyu' 'GC_Hon' 'GC_Vista'
 'GC_DNase_positive' 'GC_DNase_negative_brain' 'GC_DNase_negative_blood'
 'C_negative_heart_MK' 'C_negative_neuron_MK' 'C_negative_neuron_NP'
 'C_positive_heart_MK' 'C_positive_neuron_CD' 'C_positive_neuron_MK'
 'C_positive_neuron_NP' 'C_positive_heart_AB' 'C_SLEA'
 'GC_DNase_positive_shuffeled' 'GC_DNase_negative_brain_shuffeled'
 'GC_DNase_negative_blood_shuffeled' 'MK']


## Read all assigned oligos with the barcodes

In [ ]:
# remove the third column of assignment_barcodes.standardConfig.sorted.tsv.gz
# zcat assignment_barcodes.standardConfig.sorted.tsv.gz | head -n 10 | awk '{print $1, $2, $4}' | sort | uniq -c | sort -nr > assignment_barcodes.standardConfig_

In [95]:
# all assigned sequences with barcode
assigned_seq = '/fast/work/groups/ag_kircher/MPRA/IGVF_Y1_design/experiment/results/assignment/assignIGVFDesign/assignment_barcodes.standardConfig.sorted.tsv.gz'
# assigned_seq = '/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/experiment/standard_results/results/assignment/standardAssignIGVFDesignNoTemp/assignment_barcodes.standardConfig.sorted.tsv.gz'
# assigned_seq = '/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/experiment/standard_results/results/assignment/standardAssignIGVFDesignNoTemp/assignment_barcodes.standardConfig.sorted.tsv.gz'
assigned_seq_df = pd.read_csv(assigned_seq, sep='\t', header=None)
assigned_seq_df.columns = ['barcode', 'oligo_name', 'quality', 'number of matches']
assigned_seq_df # 7078311 rows × 4 columns


,barcode,oligo_name,quality,number of matches
0,AAAAAAAAAAACGTC,GC_Vista:eye;fb;hb;mb_hs1644_vistaElementContr...,16;270M;NM:i:0;MD:Z:270;60,5/5
1,AAAAAAAAAACAAGT,cardiac_neuro_cava_random:ALT_ZBTB20|ENSG00000...,16;270M;NM:i:0;MD:Z:270;6,7/7
2,AAAAAAAAAACACCA,cardiac_neuro_cava_random:ALT_FTO|ENSG00000140...,16;270M;NM:i:0;MD:Z:270;6,10/10
3,AAAAAAAAAACCTCG,cardiac_neuro_cava_random:REF_FKRP|ENSG0000018...,16;270M;NM:i:0;MD:Z:270;6,7/7
4,AAAAAAAAAAGCTGG,cardiac_neuro_cava_random:ALT_RAD51B|ENSG00000...,16;270M;NM:i:0;MD:Z:270;6,8/8
...,...,...,...,...
7078306,TTTTTTTTTTGACGA,cardiac_neuro_cava_random:ALT_ACTN2|ENSG000000...,16;270M;NM:i:0;MD:Z:270;6,11/11
7078307,TTTTTTTTTTGCACA,cardiac_neuro_cava_random:ALT_CARD11|ENSG00000...,16;270M;NM:i:0;MD:Z:270;6,25/26
7078308,TTTTTTTTTTGCACC,cardiac_neuro_cava_random:ALT_KRAS|ENSG0000013...,16;214M1I56M;NM:i:1;MD:Z:270;5,6/7
7078309,TTTTTTTTTTGCTAA,cardiac_neuro_cava_random:REF_NR4A2|ENSG000001...,16;270M;NM:i:0;MD:Z:270;6,6/6


In [96]:
# How many barcodes and sequqnces are duplicated?
barcode_counts = assigned_seq_df['barcode'].value_counts()
assigned_oligo_counts = assigned_seq_df['oligo_name'].value_counts()

In [97]:
# check if there are barcodes with more than one assigned sequence
print(barcode_counts[barcode_counts > 1]) # empty array -> no barcode is assigned to more than one sequence

# sequences: 
# len(assigned_oligo_counts[assigned_oligo_counts > 1]) # 74594
print(assigned_oligo_counts[assigned_oligo_counts > 1]) # assigned_oligo_counts (len: 75131) sequences with more than 1 barcode assigned: 74594

Series([], Name: count, dtype: int64)
oligo_name
GC_GABA_Chengyu:GABA|chr10:26798829-26799098|-|1.62                                                                                      1066
MK:tile_47615|chr14-29242851+29243121|reference                                                                                           893
GC_GABA_Chengyu:GABA|chr10:26798799-26799068|-|1.71                                                                                       860
MK:tile_47607|chr13-80235477+80235747|reference                                                                                           853
MK:tile_47617|chr14-29242931+29243201|reference                                                                                           848
                                                                                                                                         ... 
cardiac_neuro_cava_random:ALT_CNOT3|ENSG00000088038.20|EH38E3316446_fwd_tile1-1_CNOT3|ENSG000000880

In [9]:
# left join from all_seq_df to assigned_seq_df on oligo_name (wrong direction for checking which oligos are only in reference)
merged_df = pd.merge(assigned_seq_df, all_seq_df, on='oligo_name', how='left')

In [10]:
# how many rows have no label?
merged_df

,barcode,oligo_name,quality,number of matches,label
0,AAAAAAAAAAACGTC,GC_Vista:eye;fb;hb;mb_hs1644_vistaElementContr...,16;270M;NM:i:0;MD:Z:270;60,5/5,GC_Vista
1,AAAAAAAAAACAAGT,cardiac_neuro_cava_random:ALT_ZBTB20|ENSG00000...,16;270M;NM:i:0;MD:Z:270;6,7/7,cardiac_neuro_cava_random
2,AAAAAAAAAACACCA,cardiac_neuro_cava_random:ALT_FTO|ENSG00000140...,16;270M;NM:i:0;MD:Z:270;6,10/10,cardiac_neuro_cava_random
3,AAAAAAAAAACCTCG,cardiac_neuro_cava_random:REF_FKRP|ENSG0000018...,16;270M;NM:i:0;MD:Z:270;6,7/7,cardiac_neuro_cava_random
4,AAAAAAAAAAGCTGG,cardiac_neuro_cava_random:ALT_RAD51B|ENSG00000...,16;270M;NM:i:0;MD:Z:270;6,8/8,cardiac_neuro_cava_random
...,...,...,...,...,...
7078306,TTTTTTTTTTGACGA,cardiac_neuro_cava_random:ALT_ACTN2|ENSG000000...,16;270M;NM:i:0;MD:Z:270;6,11/11,cardiac_neuro_cava_random
7078307,TTTTTTTTTTGCACA,cardiac_neuro_cava_random:ALT_CARD11|ENSG00000...,16;270M;NM:i:0;MD:Z:270;6,25/26,cardiac_neuro_cava_random
7078308,TTTTTTTTTTGCACC,cardiac_neuro_cava_random:ALT_KRAS|ENSG0000013...,16;214M1I56M;NM:i:1;MD:Z:270;5,6/7,cardiac_neuro_cava_random
7078309,TTTTTTTTTTGCTAA,cardiac_neuro_cava_random:REF_NR4A2|ENSG000001...,16;270M;NM:i:0;MD:Z:270;6,6/6,cardiac_neuro_cava_random


In [98]:
# left join on reference by oligo_name (to check which oligos are only in reference)
all_seq_with_assignment = all_seq_df.merge(assigned_seq_df, on='oligo_name', how='left') # bwa: 7083395 
all_seq_with_assignment

,oligo_name,label,barcode,quality,number of matches
0,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random,AACATCATGGCTGAC,16;270M;NM:i:0;MD:Z:270;60,8/8
1,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random,AACCCACAATACAGG,16;270M;NM:i:0;MD:Z:270;60,9/9
2,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random,AACTTCAGCAAACCG,16;270M;NM:i:0;MD:Z:270;60,26/26
3,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random,AATTTACAGCATTCG,16;270M;NM:i:0;MD:Z:270;60,3/3
4,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random,ACACGCTGAATATAG,16;270M;NM:i:0;MD:Z:270;60,3/3
...,...,...,...,...,...
7083390,MK:tile_14356|chr15-67031618+67031887|scramble...,MK,TTAGGTTCAACCCGG,16;270M;NM:i:12;MD:Z:42A2C2T24C1T7T8C4G12C0T2A...,9/9
7083391,MK:tile_14356|chr15-67031618+67031887|scramble...,MK,TTATAGCATACCTAC,16;270M;NM:i:15;MD:Z:41T0A5T7C6C9C6T3A3T16G1T2...,7/7
7083392,MK:tile_14356|chr15-67031618+67031887|scramble...,MK,TTCTAAGCTTGACGC,16;270M;NM:i:10;MD:Z:41T6T8T5C7T1C6T2T16T10T15...,6/6
7083393,MK:tile_14356|chr15-67031618+67031887|scramble...,MK,TTTAAATCCTTGATA,16;270M;NM:i:11;MD:Z:42A13C16C3C2T24G0C3C0T2A4...,6/6


In [40]:
# how many rows of all_seq_with_assignment have no label
print(len(all_seq_with_assignment[all_seq_with_assignment['barcode'].isna()])) # 5084 rows

# get only these sequence names and store them in a tsv file (name and label)
missing_sequences = all_seq_with_assignment[all_seq_with_assignment['barcode'].isna()][['oligo_name', 'label']]
# missing_sequences["oligo_name"].unique().shape # all are unique
# ! go through df and count which sequences are missing 

5084


In [104]:
bwa_missing_seqs = all_seq_with_assignment[all_seq_with_assignment['barcode'].isna()]
bwa_missing_seqs

,oligo_name,label,barcode,quality,number of matches
1725,cardiac_neuro_cava_random:PRDM16|ENSG000001426...,cardiac_neuro_cava_random,NaN,NaN,NaN
3539,cardiac_neuro_cava_random:SZT2|ENSG00000198198...,cardiac_neuro_cava_random,NaN,NaN,NaN
8201,cardiac_neuro_cava_random:ST3GAL3|ENSG00000126...,cardiac_neuro_cava_random,NaN,NaN,NaN
13209,cardiac_neuro_cava_random:NOS1AP|ENSG000001989...,cardiac_neuro_cava_random,NaN,NaN,NaN
31540,cardiac_neuro_cava_random:RERE|ENSG00000142599...,cardiac_neuro_cava_random,NaN,NaN,NaN
...,...,...,...,...,...
7028722,MK:tile_985|chr1-33363966+33364235|LC28t6,MK,NaN,NaN,NaN
7028723,MK:tile_985|chr1-33363966+33364235|LC28t7,MK,NaN,NaN,NaN
7028725,MK:tile_985|chr1-33363966+33364235|LC28t9,MK,NaN,NaN,NaN
7068818,MK:tile_30307|chr3-171305106+171305375|scrambl...,MK,NaN,NaN,NaN


In [41]:
all_seq_with_assignment

,oligo_name,label,barcode,quality,number of matches
0,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random,AACATCATGGCTGAC,16;270M;NM:i:0;MD:Z:270;60,8/8
1,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random,AACCCACAATACAGG,16;270M;NM:i:0;MD:Z:270;60,9/9
2,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random,AACTTCAGCAAACCG,16;270M;NM:i:0;MD:Z:270;60,26/26
3,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random,AATTTACAGCATTCG,16;270M;NM:i:0;MD:Z:270;60,3/3
4,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random,ACACGCTGAATATAG,16;270M;NM:i:0;MD:Z:270;60,3/3
...,...,...,...,...,...
7083390,MK:tile_14356|chr15-67031618+67031887|scramble...,MK,TTAGGTTCAACCCGG,16;270M;NM:i:12;MD:Z:42A2C2T24C1T7T8C4G12C0T2A...,9/9
7083391,MK:tile_14356|chr15-67031618+67031887|scramble...,MK,TTATAGCATACCTAC,16;270M;NM:i:15;MD:Z:41T0A5T7C6C9C6T3A3T16G1T2...,7/7
7083392,MK:tile_14356|chr15-67031618+67031887|scramble...,MK,TTCTAAGCTTGACGC,16;270M;NM:i:10;MD:Z:41T6T8T5C7T1C6T2T16T10T15...,6/6
7083393,MK:tile_14356|chr15-67031618+67031887|scramble...,MK,TTTAAATCCTTGATA,16;270M;NM:i:11;MD:Z:42A13C16C3C2T24G0C3C0T2A4...,6/6


In [42]:
# return a list of all oligo_name's where barcode is NaN
missing_seqs = all_seq_with_assignment.loc[all_seq_with_assignment['barcode'].isna()] # 5084

In [43]:
# drop all columns except oligo_name and label
missing_seqs = missing_seqs.drop(['barcode', 'quality', 'number of matches'], axis=1)


In [44]:
# label groups with missing sequences:
label_groups_w_missing = missing_seqs['label'].unique()
len(label_groups_w_missing)
# label groups without missing sequences: find all labels that are not in label_groups_w_missing
label_groups_no_missing = [x for x in all_seq_df['label'].unique() if x not in label_groups_w_missing]
print(len(label_groups_no_missing)) # 8
label_groups_no_missing
# ['GC_Liang',
#  'GC_Mohlke',
#  'C_positive_heart_CAD',
#  'GC_Cort_Chengyu',
#  'GC_DNase_positive',
#  'GC_DNase_negative_brain',
#  'GC_DNase_negative_blood',
#  'GC_DNase_negative_brain_shuffeled']


8


['GC_Liang',
 'GC_Mohlke',
 'C_positive_heart_CAD',
 'GC_Cort_Chengyu',
 'GC_DNase_positive',
 'GC_DNase_negative_brain',
 'GC_DNase_negative_blood',
 'GC_DNase_negative_brain_shuffeled']

In [130]:
# plot the counts per label

# # check for duplicated oligo_names in missing_seqs
# missing_seqs['oligo_name'].value_counts() # all are unique
sorted_missing_seqs = missing_seqs.groupby('label').count().sort_values(by='oligo_name', ascending=False)
ax = sorted_missing_seqs.plot(kind='bar', figsize=(20, 6))
for p in ax.patches:
    ax.annotate(str(p.get_height()), (p.get_x() * 1.005, (p.get_height() + 15) * 1.005))

oligo_name
cardiac_neuro_cava_random:PRDM16|ENSG00000142611.17|EH38E2779764_fwd_tile1-1                                                                                                                   1
cardiac_neuro_cava_random:ALT_TCAP|ENSG00000173991.6|EH38E1860996_fwd_tile1-1_NEUROD2|ENSG00000171532.5|EH38E1860996|17-39643319-C-T~TCAP|ENSG00000173991.6|EH38E1860996|17-39643319-C-T       1
cardiac_neuro_cava_random:ALT_NEUROD2|ENSG00000171532.5|EH38E3221677_rev_tile1-1_NEUROD2|ENSG00000171532.5|EH38E3221677|17-39645162-T-C~TCAP|ENSG00000173991.6|EH38E3221677|17-39645162-T-C    1
cardiac_neuro_cava_random:ALT_TCAP|ENSG00000173991.6|EH38E3221677_fwd_tile1-1_NEUROD2|ENSG00000171532.5|EH38E3221677|17-39645162-T-C~TCAP|ENSG00000173991.6|EH38E3221677|17-39645162-T-C       1
cardiac_neuro_cava_random:ALT_NEUROD2|ENSG00000171532.5|EH38E1860996_rev_tile1-1_NEUROD2|ENSG00000171532.5|EH38E1860996|17-39643468-G-A~TCAP|ENSG00000173991.6|EH38E1860996|17-39643468-G-A    1
                        

## Which sequences are this exactly?
- Perpare list of unmapped sequenes
- Take two sequences by random and match them to the reference
- Write function which produces fasta file for the unmapped sequences
  - Iterate over reference.fa and take only the list of the unmapped sequences

In [22]:
# missing_seqs['oligo_name']
import yaml
# load reference fasta specified in config
config_path = '/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/experiment/config.yaml'
assignment_name = 'assignIGVFDesignNoTemp'

with open(config_path) as conf:
    config = yaml.load(conf, Loader=yaml.FullLoader)
    conf.close()
ref_fasta_path = config['assignments'][assignment_name]['reference']




In [23]:
len(missing_seqs['oligo_name'].values.tolist()) # 5084

5084

In [24]:
# subsampled data (400 lines)
# ref_fasta_path = '/data/gpfs-1/users/kisa11_c/work/coding/tmp_data/tmp_ref_no_dup.fa'
missing_seqs_list = missing_seqs['oligo_name'].values.tolist()
missing_seq_dict = prepare_missing_seq_dict(ref_fasta_path, missing_seqs_list)

NameError: name 'prepare_missing_seq_dict' is not defined

In [ ]:
len(missing_seq_dict)
missing_seq_dict.values()

{'>cardiac_neuro_cava_random:PRDM16|ENSG00000142611.17|EH38E2779764_fwd_tile1-1': 'AGGACCGGATCAACTGCCGCCCAGGAGCTCTCGTGCATCCACTCTGGTCCTCCGGTCCCGGCTGCGCCTCTTGCACCAGGCTGGGGCAGGGATTACCAGCCGCACGCAGGCTGCGGGAACCCCCTTTGTCTGGCTTTCGGCGGAGTCGGCAGAGTTCCTTCCTTCTGGGCTAATGCCCAGTTTAATTGTACATCCCATTGTGTCGTCTCTGTTCAATCATGTTCAAAAATACCTACGTCCACTCCGTTCCCATTTAGATCTCTCTAAAGTCCATTCCGGCTTATCCATTGCGTGAACCGA',
 '>cardiac_neuro_cava_random:SZT2|ENSG00000198198.17|EH38E2807306_fwd_tile1-1': 'AGGACCGGATCAACTGGGGGCGTGTGGTGGGTGGGGGGTGGGTGTTGCTAATTTAGACTGAGTGGCCAGGAAAAGCCTCACCAGGGAGGTGACAGATAAGCCGAGATCTAAATGGCAAGAAGGAATGAGTCACATGAAGACCTACAGTCAGAGCAATCCAGGGCAAAGGCAAAGTGCAAAGATAGCACATTTGGCATATCTATGGCACAGAAAGAAGGCCTGTGCGGATGAAGGGTGATAAACTGGCATGAGAATCATAAGAGATGAGAATGGCAAGGCAAGGAGCATTGCGTGAACCGA',
 '>cardiac_neuro_cava_random:ST3GAL3|ENSG00000126091.21|EH38E2807766_fwd_tile1-1': 'AGGACCGGATCAACTCCCCAACCTCTCTCACGTACACCTGCGTGTTCATGTACACATACATGTACATAGCATCCCTTGAGCTGTCCTCACCTGACCTACTGAACTCTGAGGTGGTCCGGGTCCCCCAGAGATGGCTGGGTAGCTGG

In [ ]:
## fastq files
# /data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/experiment/results/assignment/assignIGVFDesignNoTemp/fastq/merge_split0.join.fastq.gz

merged_reads = ["merge_split0.join.fastq.gz", "merge_split13.join.fastq.gz", "merge_split18.join.fastq.gz", "merge_split22.join.fastq.gz", "merge_split27.join.fastq.gz", "merge_split5.join.fastq.gz", "merge_split1.join.fastq.gz", "merge_split14.join.fastq.gz", "merge_split19.join.fastq.gz", "merge_split23.join.fastq.gz", "merge_split28.join.fastq.gz", "merge_split6.join.fastq.gz", "merge_split10.join.fastq.gz", "merge_split15.join.fastq.gz", "merge_split2.join.fastq.gz", "merge_split24.join.fastq.gz", "merge_split29.join.fastq.gz", "merge_split7.join.fastq.gz", "merge_split11.join.fastq.gz", "merge_split16.join.fastq.gz", "merge_split20.join.fastq.gz", "merge_split25.join.fastq.gz", "merge_split3.join.fastq.gz", "merge_split8.join.fastq.gz", "merge_split12.join.fastq.gz", "merge_split17.join.fastq.gz", "merge_split21.join.fastq.gz", "merge_split26.join.fastq.gz", "merge_split4.join.fastq.gz", "merge_split9.join.fastq.gz"]

In [ ]:
def prep_query_seq(seq, length, reverse=False):
    '''
    Prepares the query sequence for the search
    @param: sequence with full length; length: disired length; reverse: bool if reverse sequence is required
    @output: query sequence
    '''
    if reverse:
        query_seq = seq[::-1]
    return seq[:length]
        
# feature assignment exact matches 
def exact_match(fastq_file_path, seq_dict, seq_length, reverse=False):
    '''Tries to find all sequences in one read sequence. Iterates each missing sequence for an exact match. The sequence length is given by the user.'''
    seperation = ''.join(['-'] * 20)
    counter = 0
    with gzip.open(fastq_file_path, 'rt') as reads_fastq:
        with open(_output_file_path, 'w') as output_file:
            for line in reads_fastq:
                if line.startswith('@'):
                    header = line
                elif line.startswith('+'):
                    pass
                else:
                    read_seq = line
                    for header, seq in seq_dict.items():
                        query_seq = prep_query_seq(seq, seq_length, reverse)
                        if query_seq in read_seq:
                            counter += 1
                            output_file.write(f'{seperation}\n\nFound match\n')
                            output_file.write(f'In header: {header}\n')
                            output_file.write(f'match of length {seq_length} of\nquery: {header}\n\nseq:{query_seq}\n\n')
                            output_file.write(f'in sequence: {read_seq}\n')
                            output_file.write(f'{seperation}\n')
                            # print(f'{seperation}\n\nFound match\n')
                            # print('In header: ',header)
                            # print(f'match of length {seq_length} of ', query_seq)
                            # print('in sequence: ', seq)
                            # print(f'{seperation}')
            output_file.write(f'Found {counter} matches in {fastq_file_path}\n')
    print(f'Found {counter} matches')
# main

In [ ]:
    test_search_dict = {
        'search header': 'TCCCAGGTGAGATGGGGAGGTGAGTAGCAGATGATCTCGTGGAAGCTCCTCACCAACCTCCATCCTCTCAGTTCCTGGAAGATCACAGGGTGTTCTGTGAAGACTCAGAT'
    }
    # test_path = '/home/kisa/coding/scripts/test.fastq'
    merged_reads_path = '/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/experiment/results/assignment/assignIGVFDesignNoTemp/fastq/merge_split0.join.fastq.gz'
    exact_match(merged_reads_path, missing_seq_dict, 110, False)

NameError: name 'exact_match' is not defined

In [ ]:
import json 
json.dump(missing_seq_dict, open('/data/gpfs-1/users/kisa11_c/work/coding/80K_analysis/missing_seq_dict.json', 'w+'))

## Filter reference_exact for missing sequences


In [ ]:
# write missing sequences to file (.fa)
import json 
missing_seq_dict = json.load(open('/data/gpfs-1/users/kisa11_c/work/coding/80K_analysis/missing_seq_dict.json', 'r'))

In [ ]:
with open('/data/gpfs-1/users/kisa11_c/work/coding/80K_analysis/all_missing_sequences.fa', 'w') as missing_seq_file:
    for header, seq in missing_seq_dict.items():
        missing_seq_file.write(f'{header}\n{seq}\n')
    

## Script:

In [ ]:
# import:
import pandas as pd

## input:
assigned_barcodes = '/fast/work/groups/ag_kircher/MPRA/IGVF_Y1_design/experiment/results/assignment/assignIGVFDesign/assignment_barcodes.standardConfig.sorted.tsv.gz'

# all assigned sequences with barcode
assigned_seq_df = pd.read_csv(assigned_barcodes, sep='\t', header=None)

### Check bowtie results samtools idxstats from merged bam files
- filtering based on mapped reads does not make any sense
- "/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/experiment/results/assignment/assignIGVFDesignNoTempBowtie/bam/idxstats_bowtie.tsv"

In [56]:
bowtie_results = '/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/experiment/results/assignment/assignIGVFDesignNoTempBowtie/bam/idxstats_bowtie.tsv'
bowtie_results = '/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/experiment/results/assignment/assignIGVFDesignNoTempBowtie/bam/idxstats_bowtie2.tsv'
bowtie_df = pd.read_csv(bowtie_results, sep='\t', header=None)
bowtie_df.columns = ['oligo_name', 'length', 'mapped_reads', 'unmapped_reads']

In [57]:
bowtie_df

,oligo_name,length,mapped_reads,unmapped_reads
0,cardiac_neuro_cava_random:SKI|ENSG00000157933....,300,636,0
1,cardiac_neuro_cava_random:SKI|ENSG00000157933....,300,441,0
2,cardiac_neuro_cava_random:SKI|ENSG00000157933....,300,206,0
3,cardiac_neuro_cava_random:SKI|ENSG00000157933....,300,164,0
4,cardiac_neuro_cava_random:SKI|ENSG00000157933....,300,717,0
...,...,...,...,...
80211,MK:tile_6675|chr11-2374617+2374886|scramble_ne...,300,88,0
80212,MK:tile_18415|chr17-71181691+71181960|scramble...,300,2647,0
80213,MK:tile_14356|chr15-67031618+67031887|scramble...,300,431,0
80214,MK:tile_22033|chr2-71506006+71506275|scramble_...,300,0,0


In [58]:
# add label as column
bowtie_df["label"] = bowtie_df["oligo_name"].str.split(':').str[0] # 80215 + 1 
# lost because no/not enough reads found
missing_bowtie = bowtie_df[bowtie_df['mapped_reads'] < 10] # bowtie: 2635 +1 bowie2: 2110 + 1 
# remove the sequences without mapped reads
bowtie_df = bowtie_df[bowtie_df['mapped_reads'] > 9] # bowtie: >0: 78613; >9: 77580 bowtie2: >0 79092; >9 78105
bowtie_df

,oligo_name,length,mapped_reads,unmapped_reads,label
0,cardiac_neuro_cava_random:SKI|ENSG00000157933....,300,636,0,cardiac_neuro_cava_random
1,cardiac_neuro_cava_random:SKI|ENSG00000157933....,300,441,0,cardiac_neuro_cava_random
2,cardiac_neuro_cava_random:SKI|ENSG00000157933....,300,206,0,cardiac_neuro_cava_random
3,cardiac_neuro_cava_random:SKI|ENSG00000157933....,300,164,0,cardiac_neuro_cava_random
4,cardiac_neuro_cava_random:SKI|ENSG00000157933....,300,717,0,cardiac_neuro_cava_random
...,...,...,...,...,...
80209,MK:tile_24954|chr2-236620899+236621168|scrambl...,300,152,0,MK
80210,MK:tile_2240|chr1-116244322+116244591|scramble...,300,2668,0,MK
80211,MK:tile_6675|chr11-2374617+2374886|scramble_ne...,300,88,0,MK
80212,MK:tile_18415|chr17-71181691+71181960|scramble...,300,2647,0,MK


In [59]:
missing_bowtie

,oligo_name,length,mapped_reads,unmapped_reads,label
21,cardiac_neuro_cava_random:SKI|ENSG00000157933....,300,9,0,cardiac_neuro_cava_random
50,cardiac_neuro_cava_random:PRDM16|ENSG000001426...,300,7,0,cardiac_neuro_cava_random
93,cardiac_neuro_cava_random:SZT2|ENSG00000198198...,300,1,0,cardiac_neuro_cava_random
149,cardiac_neuro_cava_random:ST3GAL3|ENSG00000126...,300,5,0,cardiac_neuro_cava_random
208,cardiac_neuro_cava_random:NOS1AP|ENSG000001989...,300,8,0,cardiac_neuro_cava_random
...,...,...,...,...,...
79706,MK:tile_985|chr1-33363966+33364235|LC28t7,300,3,0,MK
79708,MK:tile_985|chr1-33363966+33364235|LC28t9,300,7,0,MK
80066,MK:tile_30307|chr3-171305106+171305375|scrambl...,300,0,0,MK
80214,MK:tile_22033|chr2-71506006+71506275|scramble_...,300,0,0,MK


In [60]:
# find the sequences that are not in the bowtie_df
missing_bowtie # 2635 + 1 (one line with * as oligo_name => filter out)
# filter out the line with *
missing_bowtie = missing_bowtie[missing_bowtie['oligo_name'] != '*'] 
# print the value count distribution of mapped_reads
missing_bowtie['mapped_reads'].value_counts() # bowtie: 9: 55, 8: 73, 7: 64 (sum: 2635) bowtie2: 9: 60, 8: 64, 7: 92 (sum: 2110)

mapped_reads
0    1123
1     254
2     158
3     120
4     100
7      92
5      80
8      64
9      60
6      59
Name: count, dtype: int64

In [61]:
missing_bowtie

,oligo_name,length,mapped_reads,unmapped_reads,label
21,cardiac_neuro_cava_random:SKI|ENSG00000157933....,300,9,0,cardiac_neuro_cava_random
50,cardiac_neuro_cava_random:PRDM16|ENSG000001426...,300,7,0,cardiac_neuro_cava_random
93,cardiac_neuro_cava_random:SZT2|ENSG00000198198...,300,1,0,cardiac_neuro_cava_random
149,cardiac_neuro_cava_random:ST3GAL3|ENSG00000126...,300,5,0,cardiac_neuro_cava_random
208,cardiac_neuro_cava_random:NOS1AP|ENSG000001989...,300,8,0,cardiac_neuro_cava_random
...,...,...,...,...,...
79705,MK:tile_985|chr1-33363966+33364235|LC28t6,300,4,0,MK
79706,MK:tile_985|chr1-33363966+33364235|LC28t7,300,3,0,MK
79708,MK:tile_985|chr1-33363966+33364235|LC28t9,300,7,0,MK
80066,MK:tile_30307|chr3-171305106+171305375|scrambl...,300,0,0,MK


In [133]:
mappend_thres = 9
missing_thres = 10


def plot_label_distribution(bowtie_results, mapped_threshold, missing_threshold=10):
    """
    Plot the distribution of labels in a df (samtools idxstats result)
    @param: bowtie_results: path to bowtie results file; mapped_threshold: threshold for mapped reads
    """
    bowtie_df = pd.read_csv(bowtie_results, sep='\t', header=None)
    bowtie_df.columns = ['oligo_name', 'length', 'mapped_reads', 'unmapped_reads']

    # add label as column
    bowtie_df["label"] = bowtie_df["oligo_name"].str.split(':').str[0]
    # lost because no/not enough reads found
    missing_bowtie = bowtie_df[bowtie_df['mapped_reads'] < missing_threshold]
    # filter out the line with *
    missing_bowtie = missing_bowtie[missing_bowtie['oligo_name'] != '*'] 

    # remove the sequences without mapped reads
    bowtie_df = bowtie_df[bowtie_df['mapped_reads'] > mapped_threshold] 

    # print number of missing sequences
    print(f"\n---------Summary----------\nNumber of missing sequences with less than {missing_threshold} is {len(missing_bowtie)} while {len(bowtie_df)} sequences have more than {mapped_threshold} mapped reads.") 

    # # plot the distribution of labels
    # sorted_missing_seqs_bowtie = missing_bowtie.groupby('label').count().sort_values(by='oligo_name', ascending=False)
    # # drop columns that are not needed (all except oligo_name and label)
    # sorted_missing_seqs_bowtie = sorted_missing_seqs_bowtie.drop(['length', 'mapped_reads', 'unmapped_reads'], axis=1)
    # ax = sorted_missing_seqs_bowtie.plot(kind='bar', figsize=(20, 6))
    # for p in ax.patches:
    #     ax.annotate(str(p.get_height()), (p.get_x() * 1.005, (p.get_height() + 15) * 1.005))
    return bowtie_df, missing_bowtie

In [134]:
bowtie_results = '/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/experiment/results/assignment/assignIGVFDesignNoTempBowtie/bam/idxstats_bowtie.tsv'
bowtie2_results = '/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/experiment/results/assignment/assignIGVFDesignNoTempBowtie/bam/idxstats_bowtie2.tsv'

bowtie_df, missing_bowtie = plot_label_distribution(bowtie_results, mappend_thres, missing_thres)
bowtie2_df, missing_bowtie2 = plot_label_distribution(bowtie2_results, mappend_thres, missing_thres)


---------Summary----------
Number of missing sequences with less than 10 is 2635 while 77580 sequences have more than 9 mapped reads.
2635
2635

---------Summary----------
Number of missing sequences with less than 10 is 2110 while 78105 sequences have more than 9 mapped reads.
2110
2110


#### Solution using functions

In [ ]:
# plot the distribution of labels
sorted_missing_seqs_bowtie = missing_bowtie.groupby('label').count().sort_values(by='oligo_name', ascending=False)
# drop columns that are not needed (all except oligo_name and label)
sorted_missing_seqs_bowtie = sorted_missing_seqs_bowtie.drop(['length', 'mapped_reads', 'unmapped_reads'], axis=1)
ax = sorted_missing_seqs_bowtie.plot(kind='bar', figsize=(20, 6))
for p in ax.patches:
    ax.annotate(str(p.get_height()), (p.get_x() * 1.005, (p.get_height() + 15) * 1.005))

### Identify if the aligners have missing sequences with exact matches
- read all all_exact_match.tsv: /data/gpfs-1/users/kisa11_c/work/coding/80K_analysis/01_missing_sequences/results/match_missing_sequences/all_exact_match.tsv
- find if there is a sequence which is missing by an aligner but in the exact matches 

In [1]:
# read exact matches 
exact_matches = '/data/gpfs-1/users/kisa11_c/work/coding/80K_analysis/01_missing_sequences/results/match_missing_sequences/all_exact_match.tsv'
exact_matches_df = pd.read_csv(exact_matches, sep='\t', header=None)
# add column names
exact_matches_df.columns = ['read', 'header', 'seq', 'match_length']
exact_matches_df




NameError: name 'pd' is not defined

In [123]:
# add oligo_name: without first char (>) and split header at (last) _ but multiple _ in header 
exact_matches_df['pre_oligo_name'] = exact_matches_df['header'].str.split('_').str[0:-1].str.join('_')
exact_matches_df['oligo_name'] = exact_matches_df['pre_oligo_name'].str.lstrip('>')
exact_matches_df
# # print full oligo_name entries of some rows 
# exact_matches_df['oligo_name'].head(10).tolist()[0]
# drop pre_oligo_name column
exact_matches_df = exact_matches_df.drop(['pre_oligo_name'], axis=1)

# only unique oligo_names (drop duplicates)
exact_matches_df_no_dup = exact_matches_df.drop_duplicates(subset=['oligo_name'])
exact_matches_df_no_dup

,read,header,seq,match_length,oligo_name
0,@NB501960:812:HH53WAFX5:1:11101:21622:1110 XI:...,>cardiac_neuro_cava_random:REF_PPP2R2A|ENSG000...,ACTGCAATGCTGCTAATGGATTTTGAGTTGTGGGTACCATATACCC...,270M,cardiac_neuro_cava_random:REF_PPP2R2A|ENSG0000...
1,@NB501960:812:HH53WAFX5:1:11101:4019:1255 XI:Z...,>cardiac_neuro_cava_random:REF_ANK2|ENSG000001...,GTTTTTAGTTGTGTTTAAATCAGGAATTTCCTTACTTTTCAAAATT...,270M,cardiac_neuro_cava_random:REF_ANK2|ENSG0000014...
2,@NB501960:812:HH53WAFX5:1:11101:3848:1256 XI:Z...,>cardiac_neuro_cava_random:REF_LMNA|ENSG000001...,TCTTAAATTATCTGAATCTCTTCTGAAGACAGACCTATTAGCTTTT...,270M,cardiac_neuro_cava_random:REF_LMNA|ENSG0000016...
3,@NB501960:812:HH53WAFX5:1:11101:5659:1395 XI:Z...,>MK:newcore_110746|chr13-80235318+80235587|ref...,GCTTCTAGTCATAGACAATGGCAATTGCAGAGCAAGACTAATAACA...,270M,MK:newcore_110746|chr13-80235318+80235587|refe...
4,@NB501960:812:HH53WAFX5:1:11101:23682:1535 XI:...,>cardiac_neuro_cava_random:ALT_RAI1|ENSG000001...,GAGCAACGCAGGAGAGCAGAAAGGCGAGTGCCCTGGGCTCAGGGGG...,270M,cardiac_neuro_cava_random:ALT_RAI1|ENSG0000010...
...,...,...,...,...,...
1679066,@NB501960:812:HH53WAFX5:4:11412:15223:3491 XI:...,>cardiac_neuro_cava_random:ALT_PIGQ|ENSG000000...,TTTTGGACAAGTGAGTTAATGTCCACCTCCAGTCCTTAGCACTTGC...,270M,cardiac_neuro_cava_random:ALT_PIGQ|ENSG0000000...
1686178,@NB501960:812:HH53WAFX5:4:21411:3048:11026 XI:...,>cardiac_neuro_cava_random:REF_IGLV3-25|ENSG00...,TGCCCAGCGAGACCTGAGTGGTTTTTTTTTTCATTTGTGTGAAATG...,270M,cardiac_neuro_cava_random:REF_IGLV3-25|ENSG000...
1688461,@NB501960:812:HH53WAFX5:4:21511:21490:14960 XI...,>cardiac_neuro_cava_random:REF_RERE|ENSG000001...,GCCCACGCCCTCTGCCACTGCAGTTCCCCCACAGGGCTCCCCCACG...,270M,cardiac_neuro_cava_random:REF_RERE|ENSG0000014...
1752030,@NB501960:812:HH53WAFX5:1:11101:10956:17409 XI...,>C_SLEA:SLEA_hg18:chr2:210861483-210861650|57:...,TAGGCTTCTCAAAAGTTATTTTTAAAGACTGAGGAATTAGGCACCT...,270M,C_SLEA:SLEA_hg18:chr2:210861483-210861650|57:V...


In [ ]:
### with a function: (!TODO )



In [126]:
# bowtie: 
# left join of exact_matches_df_no_dup and missing_bowtie on oligo_name
missing_bowtie = missing_bowtie.drop(['label'], axis=1)
bowtie_missing = missing_bowtie.merge(exact_matches_df_no_dup, on='oligo_name', how='left')
bowtie_missing

# get the oligo_names non nan
# bowtie2_missing[bowtie2_missing['read'].notna()]['oligo_name'].tolist()
bowtie_missing[bowtie_missing['read'].notna()]
print("Number of missing reads with exact match in design: ", len(bowtie_missing[bowtie_missing['read'].notna()]))
# check if bowtie_missing has 0 in matched still
bowtie_missing[bowtie_missing['read'].notna()]['mapped_reads'].value_counts() # no 0 in mapped reads
# Number of missing reads with exact match in design:  66
# No 0 in mapped reads

# bowtie2:
# left join of exact_matches_df_no_dup and missing_bowtie on oligo_name
missing_bowtie2 = missing_bowtie2.drop(['label'], axis=1)
bowtie2_missing = missing_bowtie2.merge(exact_matches_df_no_dup, on='oligo_name', how='left')
bowtie2_missing

# get the oligo_names non nan
# bowtie2_missing[bowtie2_missing['read'].notna()]['oligo_name'].tolist()
bowtie2_missing[bowtie2_missing['read'].notna()]
print("Number of missing reads with exact match in design: ", len(bowtie2_missing[bowtie2_missing['read'].notna()]))
# check if bowtie2_missing has 0 in matched still
bowtie2_missing[bowtie2_missing['read'].notna()]['mapped_reads'].value_counts() # no 0 in mapped reads
# Number of missing reads with exact match in design:  40
# No 0 in mapped reads



Number of missing reads with exact match in design:  66
Number of missing reads with exact match in design:  40


mapped_reads
1    10
2    10
7     6
4     5
9     3
6     3
5     2
8     1
Name: count, dtype: int64

In [110]:
bowtie2_missing

,oligo_name,length,mapped_reads,unmapped_reads,read,header,seq,match_length
0,cardiac_neuro_cava_random:SKI|ENSG00000157933....,300,9,0,NaN,NaN,NaN,NaN
1,cardiac_neuro_cava_random:PRDM16|ENSG000001426...,300,7,0,NaN,NaN,NaN,NaN
2,cardiac_neuro_cava_random:SZT2|ENSG00000198198...,300,1,0,NaN,NaN,NaN,NaN
3,cardiac_neuro_cava_random:ST3GAL3|ENSG00000126...,300,5,0,NaN,NaN,NaN,NaN
4,cardiac_neuro_cava_random:NOS1AP|ENSG000001989...,300,8,0,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
2122,MK:tile_985|chr1-33363966+33364235|LC28t6,300,4,0,NaN,NaN,NaN,NaN
2123,MK:tile_985|chr1-33363966+33364235|LC28t7,300,3,0,NaN,NaN,NaN,NaN
2124,MK:tile_985|chr1-33363966+33364235|LC28t9,300,7,0,NaN,NaN,NaN,NaN
2125,MK:tile_30307|chr3-171305106+171305375|scrambl...,300,0,0,NaN,NaN,NaN,NaN


In [109]:
bwa_missing_seqs

,oligo_name,label,barcode,quality,number of matches
1725,cardiac_neuro_cava_random:PRDM16|ENSG000001426...,cardiac_neuro_cava_random,NaN,NaN,NaN
3539,cardiac_neuro_cava_random:SZT2|ENSG00000198198...,cardiac_neuro_cava_random,NaN,NaN,NaN
8201,cardiac_neuro_cava_random:ST3GAL3|ENSG00000126...,cardiac_neuro_cava_random,NaN,NaN,NaN
13209,cardiac_neuro_cava_random:NOS1AP|ENSG000001989...,cardiac_neuro_cava_random,NaN,NaN,NaN
31540,cardiac_neuro_cava_random:RERE|ENSG00000142599...,cardiac_neuro_cava_random,NaN,NaN,NaN
...,...,...,...,...,...
7028722,MK:tile_985|chr1-33363966+33364235|LC28t6,MK,NaN,NaN,NaN
7028723,MK:tile_985|chr1-33363966+33364235|LC28t7,MK,NaN,NaN,NaN
7028725,MK:tile_985|chr1-33363966+33364235|LC28t9,MK,NaN,NaN,NaN
7068818,MK:tile_30307|chr3-171305106+171305375|scrambl...,MK,NaN,NaN,NaN


In [8]:
exact_matches_df

NameError: name 'exact_matches_df' is not defined

In [117]:
bwa_missing_seqs

,oligo_name,label,barcode,quality,number of matches
1725,cardiac_neuro_cava_random:PRDM16|ENSG000001426...,cardiac_neuro_cava_random,NaN,NaN,NaN
3539,cardiac_neuro_cava_random:SZT2|ENSG00000198198...,cardiac_neuro_cava_random,NaN,NaN,NaN
8201,cardiac_neuro_cava_random:ST3GAL3|ENSG00000126...,cardiac_neuro_cava_random,NaN,NaN,NaN
13209,cardiac_neuro_cava_random:NOS1AP|ENSG000001989...,cardiac_neuro_cava_random,NaN,NaN,NaN
31540,cardiac_neuro_cava_random:RERE|ENSG00000142599...,cardiac_neuro_cava_random,NaN,NaN,NaN
...,...,...,...,...,...
7028722,MK:tile_985|chr1-33363966+33364235|LC28t6,MK,NaN,NaN,NaN
7028723,MK:tile_985|chr1-33363966+33364235|LC28t7,MK,NaN,NaN,NaN
7028725,MK:tile_985|chr1-33363966+33364235|LC28t9,MK,NaN,NaN,NaN
7068818,MK:tile_30307|chr3-171305106+171305375|scrambl...,MK,NaN,NaN,NaN


In [122]:
# how many unique oligo_names in exact_matches_df
exact_matches_df['oligo_name'].value_counts()
# len(exact_matches_df['oligo_name'].unique()) # 2800

oligo_name
GC_GABA_Chengyu:GABA|chr10:26798769-26799038|+|1.64                                                                                                                                          9818
MK:newcore_110746|chr13-80235318+80235587|reference                                                                                                                                          9792
MK:tile_38662|chr7-20964368+20964637|reference                                                                                                                                               9746
GC_GABA_Chengyu:GABA|chr10:26798784-26799053|+|1.64                                                                                                                                          7683
GC_GABA_Chengyu:GABA|chr10:26798844-26799113|+|1.71                                                                                                                                          7503
                   

In [129]:
# bwa:
bwa_missing_mod = bwa_missing_seqs.drop(['barcode', 'quality', 'number of matches'], axis=1)
bwa_missing_mod
bwa_missing = bwa_missing_mod.merge(exact_matches_df_no_dup, on='oligo_name', how='left')
bwa_missing

# get the oligo_names non nan
# bowtie2_missing[bowtie2_missing['read'].notna()]['oligo_name'].tolist()
bwa_missing[bwa_missing['read'].notna()]
print("Number of missing reads with exact match in design: ", len(bwa_missing[bwa_missing['read'].notna()]))
# check if bwa_missing has 0 in matched still
bwa_missing[bwa_missing['read'].notna()]['label'].value_counts() # no 0 in mapped reads
# Number of missing reads with exact match in design:  66
# No 0 in mapped reads


Number of missing reads with exact match in design:  2800


label
cardiac_neuro_cava_random    2639
GC_Mendelian_variants          58
MK                             54
GC_GABA_Chengyu                 9
GC_Glut_Chengyu                 9
GC_Kircher                      3
GC_Atrial_fib                   3
C_negative_neuron_MK            3
C_negative_neuron_NP            3
C_positive_neuron_MK            3
C_positive_neuron_NP            3
C_SLEA                          3
GC_Vista                        3
GC_Selvarajan                   2
GC_Hon                          2
C_positive_heart_AB             2
C_positive_heart_MK             1
Name: count, dtype: int64

### Mobil analysis of missing sequences and their exact matches
1. Wie ist die menge von missing_sequences definiert?
  - Alle Oligo namen, die zwischen assignment und dem disign file verloren gegangen sind
2. Find ich alignment von den missing sequences in den bam files?
3. Ist all_exact_match wirklich richtig, weil nach dem aktuellen Stand gibt es keine missing sequences, die auch durch exacte matches gefunden wurden
  - Ja es war korrekt aber hatte an oligo header noch forw oder revc dran (removed by clipping last "_")
=> Nun konnten wir die missing sequences finden
4. Was ist der Unterschied zwischen: `/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/experiment/standard_results/results/assignment/standardAssignIGVFDesignNoTemp/barcodes_incl_other.sorted.tsv.gz` und `/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/experiment/standard_results/results/assignment/standardAssignIGVFDesignNoTemp/bam/barcodes_incl_other.bwa_view_1792.tsv`

In [7]:
## script finds all missing sequences of the current assignment: from notebook: /data/gpfs-1/users/kisa11_c/work/coding/80K_analysis/01_missing_sequences/notebooks/missing_sequences_per_label.ipynb
# notebook on 76
# import:
import pandas as pd
from Bio import SeqIO
import yaml
import pysam
# load reference fasta specified in config
config_path = '/data/gpfs-1/users/kisa11_c/work/coding/80K_analysis/01_missing_sequences/config/config.yml'

with open(config_path) as conf:
    config = yaml.load(conf, Loader=yaml.FullLoader)
    conf.close()

## input:
assigned_barcodes = '/fast/work/groups/ag_kircher/MPRA/IGVF_Y1_design/experiment/results/assignment/assignIGVFDesign/assignment_barcodes.standardConfig.sorted.tsv.gz'
all_headers = '/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/experiment/standard_results/results/assignment/standardAssignIGVFDesignNoTemp/reference/all_headers.tsv'
# # Testing local data
# assigned_barcodes = '/home/kisa/coding/80K_MPRA/80K-Analysis/05_variant_region_list/resources/assignment_barcodes.standardConfig.sorted.tsv.gz'
# all_headers = '/home/kisa/coding/80K_MPRA/80K-Analysis/05_variant_region_list/resources/all_headers.tsv'

used_for_80K = True

# all assigned sequences with barcode
assigned_seq_df = pd.read_csv(assigned_barcodes, sep='\t', header=None)
assigned_seq_df.columns = ['barcode', 'oligo_name', 'quality', 'number of matches']

# all sequences / headers in the design
all_seq_df = pd.read_csv(all_headers, sep='\t', header=None)
all_seq_df.columns = ['oligo_name']

# for 80K analysis: check if number of unique labels is 29 and throw error if not
if used_for_80K:
    # add second column which is split at first : and contains the label (otherwise label column is assumed)
    all_seq_df['label'] = all_seq_df['oligo_name'].str.split(':').str[0]
    if len(all_seq_df['label'].unique()) != 29:
        raise ValueError('Number of unique labels is not 29')

# print the length of the data frames:
print('\n----- Summary of the input files: -------')
print('Number of assigned sequences: ', len(assigned_seq_df.groupby('oligo_name')))
print('Number of all sequences in the design: ', len(all_seq_df))


# left join on reference by oligo_name (to check which oligos are only in reference)
all_seq_with_assignment = all_seq_df.merge(assigned_seq_df, on='oligo_name', how='left')

# get only the missing sequence names and store them in a tsv file (name and label) (barcode is na because left join did not find a match)
missing_sequences = all_seq_with_assignment[all_seq_with_assignment['barcode'].isna()][['oligo_name', 'label']]

# print the number of missing sequences
print('\n----- Summary of the missing sequences: -------')
print('Number of missing sequences: ', len(missing_sequences))





----- Summary of the input files: -------
Number of assigned sequences:  75131
Number of all sequences in the design:  80215

----- Summary of the missing sequences: -------
Number of missing sequences:  5084


In [61]:
missing_sequences

,oligo_name,label
1725,cardiac_neuro_cava_random:PRDM16|ENSG000001426...,cardiac_neuro_cava_random
3539,cardiac_neuro_cava_random:SZT2|ENSG00000198198...,cardiac_neuro_cava_random
8201,cardiac_neuro_cava_random:ST3GAL3|ENSG00000126...,cardiac_neuro_cava_random
13209,cardiac_neuro_cava_random:NOS1AP|ENSG000001989...,cardiac_neuro_cava_random
31540,cardiac_neuro_cava_random:RERE|ENSG00000142599...,cardiac_neuro_cava_random
...,...,...
7028722,MK:tile_985|chr1-33363966+33364235|LC28t6,MK
7028723,MK:tile_985|chr1-33363966+33364235|LC28t7,MK
7028725,MK:tile_985|chr1-33363966+33364235|LC28t9,MK
7068818,MK:tile_30307|chr3-171305106+171305375|scrambl...,MK


In [3]:
def fasta2pandasDF(fasta_file):
    """Read a fasta file but the sequence is in one line"""
    # read the fasta file with the sequences and prepare a tsv with header and sequence using biopython
    records = list(Bio.SeqIO.parse(fasta_file, "fasta"))
    design_df = pd.DataFrame(columns=['header', 'sequence'])
    header = [] 
    sequence = []
    for record in records:
        header.append(record.id)
        sequence.append(str(record.seq))

    design_df['header'] = header
    design_df['sequence'] = sequence
    return design_df

def write_fasta(sequence_df, output_path):
    """
    Write the fasta file with the header and sequence
    """
    with open(output_path, 'w') as f:
        for index, row in sequence_df.iterrows():
            f.write('>' + row['header'] + '\n' + row['sequence'] + '\n')
    return True
    
def read_raw_oligo_read_table(file_path=None):
    """
    Reads the read_oligo table 
    if file path == None: read example
    """
    if not file_path:
        reads = ["@NB501960:812:HH53WAFX5:1:11101:21622:1110 XI:", "@NB501960:812:HH53WAFX5:1:11101:4019:1255 XI:Z", "@NB501960:812:HH53WAFX5:1:11101:3848:1256 XI:Z",
        "@NB501960:812:HH53WAFX5:1:11101:3848:1256 XI:Z", "@NB501960:812:HH53WAFX5:1:11101:5659:1395 XI:Z"]

        oligo = [">cardiac_neuro_cava_random:REF_PPP2R2A|ENSG000", ">cardiac_neuro_cava_random:REF_ANK2|ENSG000001", ">cardiac_neuro_cava_random:REF_LMNA|ENSG000001",
        ">cardiac_neuro_cava_random:REF_ANK2|ENSG000001",
        ">MK:newcore_110746|chr13-80235318+80235587|ref"]

        reads_oligo_dict = {
            "read_name": reads,
            "oligo_name": oligo 
        }
        raw_read_oligo_df = pd.DataFrame(reads_oligo_dict)
    else:
        raw_read_oligo_df = pd.read_csv(file_path, sep="\t", header=None)
        raw_read_oligo_df.columns = ["read", "oligo", "sequence", "match_length"]
        # the oligo is ><oligo_name>_revc/forw (remove the > and split at last _)
        raw_read_oligo_df["oligo_name"] = raw_read_oligo_df["oligo"].str.split("_").str[:-1].str.join("_").str.lstrip(">")
    return raw_read_oligo_df


In [9]:
# write missing sequences to file (.fa)
# get sequences for missing sequences from the design fasta
design_fasta = "/fast/groups/ag_kircher/MPRA/IGVF_Y1_design/resources/association_data/design_no_duplicates_sequence_and_header.fa"

design_fasta_df = fasta2pandasDF(design_fasta)
# design_fasta_df.header.tolist()[0] # example header 'cardiac_neuro_cava_random:SKI|ENSG00000157933.11|EH38E2778476_fwd_tile1-1'

# match the sequences 
missing_with_seqs = missing_sequences.merge(design_fasta_df, left_on='oligo_name', right_on='header', how='left')
missing_with_seqs
# # are there duplicated sequences in the missing sequences?  NO! Nice
# missing_with_seqs['sequence'].value_counts() # no duplicates
# missing_with_seqs['header'].value_counts() # no duplicates
# missing_with_seqs['oligo_name'].value_counts() # no duplicates  
# # write the missing sequences to a fasta
output_missing_sequences = config["files"]["missing_sequences"]

write_fasta(missing_with_seqs, output_missing_sequences)


True

In [10]:
missing_sequences # 5084 
# how many from cardiac_neuro_cava_random?
# missing_sequences[missing_sequences.label == "cardiac_neuro_cava_random"] # 4657

# how many with exact match? (download: "/data/gpfs-1/users/kisa11_c/work/coding/80K_analysis/01_missing_sequences/results/match_missing_sequences/all_exact_match.tsv")
all_exact_match = "/data/gpfs-1/users/kisa11_c/work/coding/80K_analysis/01_missing_sequences/results/match_missing_sequences/all_exact_match.tsv"
raw_read_oligo_df = read_raw_oligo_read_table(all_exact_match) # replace None by path to all_exact_match
# raw_read_oligo_df = read_raw_oligo_read_table(None) # local test
raw_read_oligo_df

# match this table to the table of missing sequences by oligo_name (left join)
merged_read_missing_oligo_df = raw_read_oligo_df.merge(missing_sequences, how="left", on="oligo_name")

merged_read_missing_oligo_df
# # check number of not nan in label column
missing_sequence_with_exact_match = merged_read_missing_oligo_df[merged_read_missing_oligo_df["label"].notna()]
print("Number of missing reads with exact match: ", missing_sequence_with_exact_match.shape[0])
print("Results in number of missing sequences with exact match: ", len(missing_sequence_with_exact_match.groupby("oligo_name")["read"].apply(list)))


Number of missing reads with exact match:  1812292
Results in number of missing sequences with exact match:  oligo_name
C_SLEA:SLEA_hg18:chr2:210861483-210861650|57:V_HNF3ALPHA_Q6:TGTTTGCTTTG;88:V_COUPTF_Q6:CCCCCTGACCTTTGCCCCCTGCC                                                                                                                                                                                              [@NB501960:812:HH53WAFX5:1:11101:10956:17409 X...
C_SLEA:SLEA_hg18:chr2:210861483-210861650|8:V_PPARA_02:CCGGGTCATTGGGGTCAGG;30:V_HNF3ALPHA_Q6:TGTTTGCTTTG;44:V_AHRARNT_02:GGGGATCGCGTGCCAGCCC;66:V_Rxra_UP:GGCCGTGACCCCGTGAT;86:V_HNF4_Q6:AAGGTCCAG;98:V_XBP1_01:GTGATGACGTGTCCCAT;118:V_COUPTF_Q6:CCCCCTGACCTTTGCCCCCTGCC;144:V_HNF1_C:AGTTAATGATTAACCAA    [@NB501960:812:HH53WAFX5:1:21312:8501:11279 XI...
C_SLEA:SLEA_hg18:chr9:82902419-82902586|18:V_HNF4_Q6:AAGGTCCAG;30:V_Rxra_UP:GGCCGTGACCCCGTGAT;50:V_HNF1_C:AGTTAATGATTAACCAA;70:V_HNF6_Q6:CAAAATCAATAA;85:V_AHRARNT_02:GGGGATCGCGTGCC

In [5]:
left_test = missing_sequences.merge(raw_read_oligo_df, how="left", on="oligo_name")
left_test[left_test["read"].notna()].groupby("oligo_name")

#### Identify the sequences in the bam file

In [66]:
# # identify how many sequences are missing with exact match (for cardiac_neuro_cava_random: 2639)
# merged_read_missing_oligo_df # check how many sequences are there with exact match
# merged_read_missing_oligo_df[merged_read_missing_oligo_df["label"].notna()].groupby("oligo_name").count() # 2800 sequences overall

# # filter for results with lable == cardiac_neuro_cava_random
merged_read_missing_cardiac_neuro_cava_random = merged_read_missing_oligo_df[merged_read_missing_oligo_df["label"] == "cardiac_neuro_cava_random"]
merged_read_missing_cardiac_neuro_cava_random = merged_read_missing_cardiac_neuro_cava_random[merged_read_missing_cardiac_neuro_cava_random["label"].notna()]
# merged_read_missing_cardiac_neuro_cava_random.groupby("oligo_name").count() # 2639 sequences
# # are all reads unique? yes
# merged_read_missing_cardiac_neuro_cava_random["read"].value_counts() # yes Length: 1579132
# merged_read_missing_cardiac_neuro_cava_random # 1579132 
# merged_read_missing_oligo_df # 1812292 
# merged_read_missing_oligo_df["read"].value_counts() # 1812292 
# get an example sequence and the reads in a dict
# group the merged_read_missing_oligo_df by oligo_name and get the list of all reads per oligo_name
# merged_read_missing_oligo_df[merged_read_missing_oligo_df["label"].notna()].groupby("oligo_name")["read"].apply(list) # 2800 sequences

read_list_per_missing_sequence = merged_read_missing_cardiac_neuro_cava_random.groupby("oligo_name")["read"].apply(list) # 2639 sequences
missing_sequence_list_per_read = merged_read_missing_cardiac_neuro_cava_random.groupby("read")["oligo_name"].apply(list) # 1579132
# read_list_per_missing_sequence
# # make dict from list (missing_sequence: [reads])
read_dict_per_missing_sequence = read_list_per_missing_sequence.to_dict()
missing_sequence_dict_per_read = missing_sequence_list_per_read.to_dict()



In [88]:
missing_sequence_list_per_read

read
@NB501960:812:HH53WAFX5:1:11101:10000:13193 XI:Z:AATTTAGTGTTTTGA,YI:Z:AAAAAEEEEEEEEEE    [cardiac_neuro_cava_random:REF_PTPN11|ENSG0000...
@NB501960:812:HH53WAFX5:1:11101:10000:3924 XI:Z:TAGACTGTACTAAGT,YI:Z:AAAAAEEEEEEEEEE     [cardiac_neuro_cava_random:REF_FANCM|ENSG00000...
@NB501960:812:HH53WAFX5:1:11101:10002:10961 XI:Z:TTTCCGCCTTGGGTT,YI:Z:AAAAAEEEEEEEEEE    [cardiac_neuro_cava_random:REF_KCNA2|ENSG00000...
@NB501960:812:HH53WAFX5:1:11101:10004:3092 XI:Z:CACTGCTTTCACTGC,YI:Z:AAAAAEEEEE6EEEE     [cardiac_neuro_cava_random:ALT_TCAP|ENSG000001...
@NB501960:812:HH53WAFX5:1:11101:10004:6204 XI:Z:TTGCGTTGCGTGGAA,YI:Z:AAAAAEEEEEEEEEE     [cardiac_neuro_cava_random:ALT_NTHL1|ENSG00000...
                                                                                                               ...                        
@NB501960:812:HH53WAFX5:4:21612:9973:8222 XI:Z:AGCAGTGTTGAGAGA,YI:Z:/AAAAAEAEEEEEEE      [cardiac_neuro_cava_random:REF_SREBF1|ENSG0000...
@NB501960:812:HH53WAFX

In [78]:
missing_sequence_dict_per_read["@NB501960:812:HH53WAFX5:1:11101:10000:13193 XI:Z:AATTTAGTGTTTTGA,YI:Z:AAAAAEEEEEEEEEE"]

['cardiac_neuro_cava_random:REF_PTPN11|ENSG00000179295.19|EH38E3041838_fwd_tile1-1']

In [87]:
## read the merged (!!!) bam file and investigate the missing reads
merged_bam = "/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/experiment/standard_results/results/assignment/standardAssignIGVFDesignNoTemp/bam/bwa_merged.bam"

# bam_file_path = "..."
read_count = 0
samfile = pysam.AlignmentFile(merged_bam, "rb")
for read in samfile.fetch():
    # TODO: add a function which gets the read and the sequences of interest table and returns if the read is a exact matching read or not 
    # see https://pysam.readthedocs.io/en/latest/api.html#pysam.AlignedSegment for examples of pysam functionality
    # print(read)
    # print(read.query_name) # query name
    # print(read.get_forward_sequence()) # original read sequence
    # print(read.get_reference_sequence()) # reference sequence
    # print(read.tags) # tags of samfile
    # # prapare header name from query name and tags
    # # @NB501960:812:HH53WAFX5:1:11101:10000:13193 XI:Z:AATTTAGTGTTTTGA,YI:Z:AAAAAEEEEEEEEEE
    # read.tags[-1] = ('XI', 'AATTTAGTGTTTTGA,YI:Z:AAAAAEEEEEEEEEE')
    prepared_read_name = f'@{read.query_name} {read.tags[-1][0]}:Z:{read.tags[-1][1]}'
    # try to find the prepared read name in the missing_sequence_dict_per_read
    if prepared_read_name in missing_sequence_dict_per_read.keys():
        read_count += 1
        print(prepared_read_name)
samfile.close()

# for later: output_samfile.write(AlignedSegment_to_be_written)

@NB501960:812:HH53WAFX5:1:11105:9427:8075 XI:Z:CACGAGTTTACGTGA,YI:Z:AAAAAEEEEEEEEAE


In [101]:
missing_sequence_dict_per_read["@NB501960:812:HH53WAFX5:1:11101:10000:13193 XI:Z:AATTTAGTGTTTTGA,YI:Z:AAAAAEEEEEEEEEE"][0]

'cardiac_neuro_cava_random:REF_PTPN11|ENSG00000179295.19|EH38E3041838_fwd_tile1-1'

In [99]:
missing_sequence_dict_per_read["@NB501960:812:HH53WAFX5:1:11105:9427:8075 XI:Z:CACGAGTTTACGTGA,YI:Z:AAAAAEEEEEEEEAE"][0]

KeyError: 'NB501960:812:HH53WAFX5:1:11105:9427:8075 XI:Z:CACGAGTTTACGTGA,YI:Z:AAAAAEEEEEEEEAE'

In [85]:
# read_count # 1812292
# 4470992 * 18

80477856

In [49]:
# investigate read_list_per_missing_sequence
# type(read_list_per_missing_sequence) # pandas.core.series.Series
# investigate read_dict_per_missing_sequence
# type(read_dict_per_missing_sequence) # dict
# type(read_dict_per_missing_sequence['cardiac_neuro_cava_random:ADAMTS7|ENSG00000136378.15|EH38E3148911_rev_tile1-1']) # list


str

In [11]:
missing_reads_cardiac_neuro_cava = merged_read_missing_cardiac_neuro_cava_random[merged_read_missing_cardiac_neuro_cava_random["label"].notna()].groupby("oligo_name").count() # 2639 sequences
missing_reads_cardiac_neuro_cava = missing_reads_cardiac_neuro_cava.drop(["sequence", "match_length"], axis=1)
missing_reads_cardiac_neuro_cava
# get a dict of all the reads assigned to the sequences and 

,read,oligo,label
oligo_name,,,
cardiac_neuro_cava_random:ADAMTS7|ENSG00000136378.15|EH38E3148911_rev_tile1-1,75,75,75
cardiac_neuro_cava_random:ALT_ACVR2B|ENSG00000114739.14|EH38E2193307_fwd_tile1-1_ACVR2B|ENSG00000114739.14|EH38E2193307|3-38509904-C-G~SCN5A|ENSG00000183873.18|EH38E2193307|3-38509904-C-G,2530,2530,2530
cardiac_neuro_cava_random:ALT_ACVR2B|ENSG00000114739.14|EH38E2193307_fwd_tile1-1_ACVR2B|ENSG00000114739.14|EH38E2193307|3-38510036-A-G~SCN5A|ENSG00000183873.18|EH38E2193307|3-38510036-A-G,2028,2028,2028
cardiac_neuro_cava_random:ALT_ACVR2B|ENSG00000114739.14|EH38E3506605_fwd_tile1-1_ACVR2B|ENSG00000114739.14|EH38E3506605|3-38516825-C-T~SCN5A|ENSG00000183873.18|EH38E3506605|3-38516825-C-T,792,792,792
cardiac_neuro_cava_random:ALT_ADAMTS7|ENSG00000136378.15|EH38E1780780_rev_tile1-1_ADAMTS7|ENSG00000136378.15|EH38E1780780|15-78804641-C-A~MORF4L1|ENSG00000185787.15|EH38E1780780|15-78804641-C-A,401,401,401
...,...,...,...
cardiac_neuro_cava_random:SREBF1|ENSG00000072310.18|EH38E3212352_rev_tile1-1,309,309,309
cardiac_neuro_cava_random:TCAP|ENSG00000173991.6|EH38E3221636_fwd_tile1-1,219,219,219
cardiac_neuro_cava_random:TCAP|ENSG00000173991.6|EH38E3221651_fwd_tile1-1,16,16,16


#### Identify the missing sequences in the idxstats result and barcodes_incl_other.sorted.tsv.gz 

In [7]:
# load idx_stat data first and check which sequences can be found in the bam because I am not sure how to look for a specific sequence in the alignment
# idxstat_bwa_path = "/home/kisa/coding/80K_MPRA/80K-Analysis/05_variant_region_list/resources/idxstats_bwa_1201.tsv"
idxstat_bwa_path = "/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/experiment/standard_results/results/assignment/standardAssignIGVFDesignNoTemp/bam/idxstats_bwa_1201.tsv"

idxstats_bwa = pd.read_csv(idxstat_bwa_path, sep='\t', header=None)
idxstats_bwa.columns = ["oligo_name", "sequence_length", "number_mapped_reads", "number_unmapped_reads"]
idxstats_bwa

mapped_idxstats = missing_sequences.merge(idxstats_bwa, how="left", on="oligo_name")
mapped_idxstats

## filter mapped_idxstats for cardiac_neuro_cava_random and mapped_reads > 0 => 3842 rows
filtered_mapped_idxstats = mapped_idxstats[mapped_idxstats["label"] == "cardiac_neuro_cava_random"]
filtered_mapped_idxstats = filtered_mapped_idxstats[filtered_mapped_idxstats["number_mapped_reads"] > 0]

## check number of na values in mapped_reads: 5084 => no na values
mapped_idxstats[mapped_idxstats["number_mapped_reads"].notna()]


## check number of mapped_reads == 0 => 924 sequences are not found and 815 of them are from cardiac_neuro_cava_random
not_found_at_all = mapped_idxstats[mapped_idxstats["number_mapped_reads"] == 0] # 924
not_found_at_all[not_found_at_all["label"] == "cardiac_neuro_cava_random"] # 815

## find the maximal value of mapped_reads # 6889 - 4910 and get the 10 oligos with highest mapped read values
missing_example_seqs_with_mapped_reads = filtered_mapped_idxstats.sort_values(by="number_mapped_reads", ascending=False).head(10)

# load the barcodes_incl_other.sorted.tsv file
# barcodes_incl_other_path = "/home/kisa/coding/80K_MPRA/80K-Analysis/05_variant_region_list/resources/barcodes_incl_other.sorted.tsv"
barcodes_incl_other_path = "/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/experiment/standard_results/results/assignment/standardAssignIGVFDesignNoTemp/barcodes_incl_other.sorted.tsv.gz"

barcodes_incl_other_df = pd.read_csv(barcodes_incl_other_path, sep="\t", header=None)
barcodes_incl_other_df.columns = ["barcode", "oligo_name", "mapping_information"]
barcodes_incl_other_df

print("Summary of mapped and unmapped oligos:\n", barcodes_incl_other_df["barcode"].value_counts())
mapped_barcodes_incl_other = barcodes_incl_other_df[barcodes_incl_other_df["barcode"] == 1]

# check number of label (cardiac_neuro_cava_random)
mapped_barcodes_incl_other["label"] = mapped_barcodes_incl_other["oligo_name"].apply(lambda x: x.split(':')[0] if ':' in x else "no : present")

print("Summary of mapped labels:\n", mapped_barcodes_incl_other["label"].value_counts())
print("Number of sequences with cardiac_neuro_cava_random as label:\n", mapped_barcodes_incl_other[mapped_barcodes_incl_other["label"] == "cardiac_neuro_cava_random"].shape[0])

print("next step: check the bam file for the missing sequences")
# load the bam and identify the sequences of interest (missing_sequence_with_exact_match)


# import pysam

# bam_file_path = "..."
# samfile = pysam.AlignmentFile(bam_file_path, "rb")
# for read in samfile.fetch():
#     # TODO: add a function which gets the read and the sequences of interest table and returns if the read is a exact matching read or not 
#     print(read)
# samfile.close()





Summary of mapped and unmapped oligos:
 barcode
GGGGGGGGGGGGGGG    302650
TGGGGGGGGGGGGGG      2812
GTGGGGGGGGGGGGG      1237
CGGGGGGGGGGGGGG      1232
CGGTCGCCACCATGG      1050
                    ...  
GCTATATTGCCCGCG         1
GCTATATTGGTGGCG         1
GCTATATTGTAATAT         1
GCTATATTGTCCACT         1
TTTTTTTTTTTTTTT         1
Name: count, Length: 9945742, dtype: int64
Summary of mapped labels:
 Series([], Name: count, dtype: int64)
Number of sequences with cardiac_neuro_cava_random as label:
 0
next step: check the bam file for the missing sequences


In [8]:
mapped_idxstats

,oligo_name,label,sequence_length,number_mapped_reads,number_unmapped_reads
0,cardiac_neuro_cava_random:PRDM16|ENSG000001426...,cardiac_neuro_cava_random,300,7,0
1,cardiac_neuro_cava_random:SZT2|ENSG00000198198...,cardiac_neuro_cava_random,300,1,0
2,cardiac_neuro_cava_random:ST3GAL3|ENSG00000126...,cardiac_neuro_cava_random,300,18,0
3,cardiac_neuro_cava_random:NOS1AP|ENSG000001989...,cardiac_neuro_cava_random,300,17,0
4,cardiac_neuro_cava_random:RERE|ENSG00000142599...,cardiac_neuro_cava_random,300,0,0
...,...,...,...,...,...
5079,MK:tile_985|chr1-33363966+33364235|LC28t6,MK,300,9,0
5080,MK:tile_985|chr1-33363966+33364235|LC28t7,MK,300,18,0
5081,MK:tile_985|chr1-33363966+33364235|LC28t9,MK,300,10,0
5082,MK:tile_30307|chr3-171305106+171305375|scrambl...,MK,300,0,0


In [9]:
# check if /home/kisa/coding/80K_MPRA/80K-Analysis/05_variant_region_list/resources/design_no_duplicates_sequence_and_header.fa has duplicates
design_fasta = "/home/kisa/coding/80K_MPRA/80K-Analysis/05_variant_region_list/resources/design_no_duplicates_sequence_and_header.fa"

header = []
sequence = []
sequences = set()
with open(design_fasta, "r") as design_fa:
    for row in design_fa:
        if row.startswith(">"):
            header.append(row.rstrip())
        else:
            sequences.add(row.rstrip())
            sequence.append(row.rstrip())

# create design_df
design_df = pd.DataFrame({"header": header, "sequence": sequence})

print(design_df.shape)


print("Is the number of unique sequences still 80215?: ", len(sequences) == 80215)


FileNotFoundError: [Errno 2] No such file or directory: '/home/kisa/coding/80K_MPRA/80K-Analysis/05_variant_region_list/resources/design_no_duplicates_sequence_and_header.fa'

In [3]:
filtered_mapped_idxstats

NameError: name 'filtered_mapped_idxstats' is not defined

In [11]:
# match left join of barcodes_incl_other_df with the 10 sequences of interest (missing_example_seqs_with_mapped_reads)
filtered_mapped_idxstats.merge(barcodes_incl_other_df, how='left', on="oligo_name")

: 

: 

In [6]:
# Needs to have a bam file with an index
import os
import pandas as pd
import pysam
import yaml

# get the input file (bam file)

config_path = '/data/gpfs-1/users/kisa11_c/work/coding/80K_analysis/01_missing_sequences/config/config.yml'

with open(config_path) as conf:
    config = yaml.load(conf, Loader=yaml.FullLoader)
    conf.close()

# merged_bam = config['files']['merged_bam']

missing_sequence_bam = config['files']['missing_sequence_bam']

# quality measures from config (/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/experiment/standard_results/standard_config.yaml):
mapping_quality_min = config['quality']['min_mapping_quality']
alignment_start_min = config['quality']['alignment_start']['min']
alignment_start_max = config['quality']['alignment_start']['max']
sequence_length_min = config['quality']['sequence_length']['min']
sequence_length_max = config['quality']['sequence_length']['max']
missing_sequences_min_quality = os.path.join(config['general']['output_dir'], "identified_missing_sequences", "missing_sequences_min_quality.bam")        
missing_sequences_alignment_start = os.path.join(config['general']['output_dir'], "identified_missing_sequences", "missing_sequences_alignment_start.bam")
missing_sequences_alignment_end = os.path.join(config['general']['output_dir'], "identified_missing_sequences", "missing_sequences_alignment_end.bam")
missing_sequences_sequence_length_min = os.path.join(config['general']['output_dir'], "identified_missing_sequences", "missing_sequences_sequence_length_min.bam")
missing_sequences_sequence_length_max = os.path.join(config['general']['output_dir'], "identified_missing_sequences", "missing_sequences_sequence_length_max.bam")

# # Debug
# print(missing_sequences_min_quality)
# print(missing_sequences_alignment_start)
# print(missing_sequences_alignment_end)
# print(missing_sequences_sequence_length_min)
# print(missing_sequences_sequence_length_max)

read_count = 0
samfile = pysam.AlignmentFile(missing_sequence_bam, "rb")
min_quality_bam = pysam.AlignmentFile(missing_sequences_min_quality, "wb", template=samfile)
alignment_start_bam = pysam.AlignmentFile(missing_sequences_alignment_start, "wb", template=samfile)
alignment_end_bam = pysam.AlignmentFile(missing_sequences_alignment_end, "wb", template=samfile)
sequence_length_min_bam = pysam.AlignmentFile(missing_sequences_sequence_length_min, "wb", template=samfile)
sequence_length_max_bam = pysam.AlignmentFile(missing_sequences_sequence_length_max, "wb", template=samfile)

for read in samfile.fetch():
    # # implemented logic
    # if read < mapping_quality_min => write
    # if read ($4) < alignment_start_min => write
    # if read ($4) > alignment_start_max => write
    # print(read.reference_end)
    # if sequence length ($10) < sequence_length_min => write
    # if sequence length ($10) > sequence_length_max => write

    if read.mapping_quality < mapping_quality_min:
        # read_count += 1
        prepared_read_name = f'@{read.query_name} {read.tags[-1][0]}:Z:{read.tags[-1][1]}'
        # print(prepared_read_name)
        # write to min_quality_bam
        min_quality_bam.write(read)

    if read.reference_start < alignment_start_min:
        # read_count += 1
        prepared_read_name = f'@{read.query_name} {read.tags[-1][0]}:Z:{read.tags[-1][1]}'
        # print(prepared_read_name)
        # print(read.reference_start)
        # write to alignment_start
        alignment_start_bam.write(read)

    if read.reference_start > alignment_start_max:
        # read_count += 1
        prepared_read_name = f'@{read.query_name} {read.tags[-1][0]}:Z:{read.tags[-1][1]}'
        # print(prepared_read_name)
        # print(read.reference_start)
        # write to alignment_end
        alignment_end_bam.write(read)

    if len(read.query_sequence) < sequence_length_min:
        # read_count += 1
        prepared_read_name = f'@{read.query_name} {read.tags[-1][0]}:Z:{read.tags[-1][1]}'
        # print(prepared_read_name)
        # print(len(read.query_sequence))
        # write to sequence_length_min
        sequence_length_min_bam.write(read)

    if len(read.query_sequence) > sequence_length_max:
        # read_count += 1
        prepared_read_name = f'@{read.query_name} {read.tags[-1][0]}:Z:{read.tags[-1][1]}'
        # print(prepared_read_name)
        # print(len(read.query_sequence))
        # write to sequence_length_max
        sequence_length_max_bam.write(read)
    read_count += 1
    # # debug
    # if read_count == 10:
    #     break

samfile.close()
min_quality_bam.close()
alignment_start_bam.close()
alignment_end_bam.close()
sequence_length_min_bam.close()
sequence_length_max_bam.close()

In [124]:
missing_sequences_bam_list = [missing_sequence_min_quality, missing_sequence_alignment_start, missing_sequences_alignment_end, missing_sequences_sequence_length_min, missing_sequences_sequence_length_max]
missing_sequences_bam_list

['/data/gpfs-1/users/kisa11_c/work/coding/80K_analysis/01_missing_sequences/results/identified_missing_sequences/missing_sequence_min_quality.bam',
 '/data/gpfs-1/users/kisa11_c/work/coding/80K_analysis/01_missing_sequences/results/identified_missing_sequences/missing_sequence_alignment_start.bam',
 '/data/gpfs-1/users/kisa11_c/work/coding/80K_analysis/01_missing_sequences/results/identified_missing_sequences/missing_sequences_alignment_end.bam',
 '/data/gpfs-1/users/kisa11_c/work/coding/80K_analysis/01_missing_sequences/results/identified_missing_sequences/missing_sequences_sequence_length_min.bam',
 '/data/gpfs-1/users/kisa11_c/work/coding/80K_analysis/01_missing_sequences/results/identified_missing_sequences/missing_sequences_sequence_length_max.bam']

## Match new assignment back to the sequences
- load new assignment
- match the second column with all_headers

In [4]:
def fasta2pandasDF(fasta_file):
    """Read a fasta file but the sequence is in one line"""
    # read the fasta file with the sequences and prepare a tsv with header and sequence using biopython
    records = list(SeqIO.parse(fasta_file, "fasta"))
    design_df = pd.DataFrame(columns=['header', 'sequence'])
    header = [] 
    sequence = []
    for record in records:
        header.append(record.id)
        sequence.append(str(record.seq))

    design_df['header'] = header
    design_df['sequence'] = sequence
    return design_df

In [1]:
import pandas as pd
import yaml
from Bio import SeqIO
config_path = '/data/gpfs-1/users/kisa11_c/work/coding/80K_analysis/01_missing_sequences/config/filter_bc_from_bam_config.yml'

with open(config_path) as conf:
    config = yaml.load(conf, Loader=yaml.FullLoader)
    conf.close()


In [2]:
## load new assignment
new_assignment_path = config["files"]["assignment_tbl"]

# old_assignment_path = config["files"]["old_assignment_tbl"]
# old_assignment = pd.read_csv(old_assignment_path, sep="\t", header=None)
new_assignment = pd.read_csv(new_assignment_path, sep="\t", header=None) # 93159920 rows × 3 columns
new_assignment.columns = ["bc", "oligo_name", "info"]

reference = config["files"]["reference"]
reference_df = fasta2pandasDF(reference)

In [5]:
reference = config["files"]["reference"]
reference_df = fasta2pandasDF(reference)

In [8]:
old_assignment_path = config["files"]["old_assignment_tbl"]
old_assignment = pd.read_csv(old_assignment_path, sep="\t", header=None)
old_assignment.columns = ["bc", "oligo_name", "info"]
old_assignment

,bc,oligo_name,info
0,AAAAAAAAAAAAAAA,cardiac_neuro_cava_random:ALT_ANK3|ENSG0000015...,16;270M;NM:i:1;MD:Z:190C79;6
1,AAAAAAAAAAACGGG,cardiac_neuro_cava_random:ALT_TNNT2|ENSG000001...,16;270M;NM:i:9;MD:Z:82C90A1A10A5C9T21A8T20C15;5
2,AAAAAAAAAAACGTC,GC_Vista:eye;fb;hb;mb_hs1644_vistaElementContr...,16;270M;NM:i:0;MD:Z:270;60
3,AAAAAAAAAAACGTC,GC_Vista:eye;fb;hb;mb_hs1644_vistaElementContr...,16;270M;NM:i:0;MD:Z:270;60
4,AAAAAAAAAAACGTC,GC_Vista:eye;fb;hb;mb_hs1644_vistaElementContr...,16;270M;NM:i:0;MD:Z:270;60
...,...,...,...
61096514,GCTGCTAAGTTACTC,cardiac_neuro_cava_random:REF_WWOX|ENSG0000018...,16;270M;NM:i:6;MD:Z:78T68A2A9A0T6T101;3
61096515,GCTGCTAATAAAATG,cardiac_neuro_cava_random:ALT_DPYSL2|ENSG00000...,16;270M;NM:i:0;MD:Z:270;6
61096516,GCTGCTAATAAAATG,cardiac_neuro_cava_random:ALT_DPYSL2|ENSG00000...,16;270M;NM:i:0;MD:Z:270;6
61096517,GCTGCTAATAAAATG,cardiac_neuro_cava_random:ALT_DPYSL2|ENSG00000...,16;270M;NM:i:0;MD:Z:270;6


In [7]:
old_assignment

,0,1,2
0,AAAAAAAAAAAAAAA,cardiac_neuro_cava_random:ALT_ANK3|ENSG0000015...,16;270M;NM:i:1;MD:Z:190C79;6
1,AAAAAAAAAAACGGG,cardiac_neuro_cava_random:ALT_TNNT2|ENSG000001...,16;270M;NM:i:9;MD:Z:82C90A1A10A5C9T21A8T20C15;5
2,AAAAAAAAAAACGTC,GC_Vista:eye;fb;hb;mb_hs1644_vistaElementContr...,16;270M;NM:i:0;MD:Z:270;60
3,AAAAAAAAAAACGTC,GC_Vista:eye;fb;hb;mb_hs1644_vistaElementContr...,16;270M;NM:i:0;MD:Z:270;60
4,AAAAAAAAAAACGTC,GC_Vista:eye;fb;hb;mb_hs1644_vistaElementContr...,16;270M;NM:i:0;MD:Z:270;60
...,...,...,...
61096514,GCTGCTAAGTTACTC,cardiac_neuro_cava_random:REF_WWOX|ENSG0000018...,16;270M;NM:i:6;MD:Z:78T68A2A9A0T6T101;3
61096515,GCTGCTAATAAAATG,cardiac_neuro_cava_random:ALT_DPYSL2|ENSG00000...,16;270M;NM:i:0;MD:Z:270;6
61096516,GCTGCTAATAAAATG,cardiac_neuro_cava_random:ALT_DPYSL2|ENSG00000...,16;270M;NM:i:0;MD:Z:270;6
61096517,GCTGCTAATAAAATG,cardiac_neuro_cava_random:ALT_DPYSL2|ENSG00000...,16;270M;NM:i:0;MD:Z:270;6


In [16]:
# check the assignment file:
# merge it 

# old_assigned_sequences = reference_df.merge(old_assignment, left_on='header', right_on='oligo_name', how='left')
# old_assigned_sequences # 61032987 rows × 5 columns
# # get the notna rows 
# old_assigned_sequences = old_assigned_sequences[old_assigned_sequences["bc"].notna()]
# groupby header to get all the sequences found
old_bc_count_table = old_assigned_sequences.groupby("header").count() # 75497 

In [20]:
old_bc_count_table[old_bc_count_table["bc"]>=5].shape[0] # 74858
old_bc_count_table[old_bc_count_table["bc"]>=10].shape[0] # 74516

74516

In [ ]:
new_assignment

In [6]:
# left join of new_assignment and reference_df on oligo_name and header
assigned_sequences = reference_df.merge(new_assignment, left_on='header', right_on='oligo_name', how='left')
assigned_sequences

,header,sequence,bc,oligo_name,info
0,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTAAGAATACAAGTAACTGATGAATGAAGGGGG...,CAGTATGAAGGTCGA,cardiac_neuro_cava_random:SKI|ENSG00000157933....,15;270M;NM:i:0;MD:Z:270;60
1,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTAAGAATACAAGTAACTGATGAATGAAGGGGG...,GCGAGTTACGACCCC,cardiac_neuro_cava_random:SKI|ENSG00000157933....,15;270M;NM:i:0;MD:Z:270;60
2,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTAAGAATACAAGTAACTGATGAATGAAGGGGG...,GGGGGGTGGTTGATA,cardiac_neuro_cava_random:SKI|ENSG00000157933....,15;270M;NM:i:0;MD:Z:270;60
3,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTAAGAATACAAGTAACTGATGAATGAAGGGGG...,GACCTAAGCCGAATG,cardiac_neuro_cava_random:SKI|ENSG00000157933....,15;270M;NM:i:1;MD:Z:163C106;60
4,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTAAGAATACAAGTAACTGATGAATGAAGGGGG...,AGTTACGGAGATTTC,cardiac_neuro_cava_random:SKI|ENSG00000157933....,15;270M;NM:i:0;MD:Z:270;60
...,...,...,...,...,...
93061711,MK:tile_14356|chr15-67031618+67031887|scramble...,AGGACCGGATCAACTTGAAGCCCCTGATTCTGTTAGAATAAGGTTA...,GTGAGAGCTGTGCCC,MK:tile_14356|chr15-67031618+67031887|scramble...,15;270M;NM:i:0;MD:Z:270;60
93061712,MK:tile_14356|chr15-67031618+67031887|scramble...,AGGACCGGATCAACTTGAAGCCCCTGATTCTGTTAGAATAAGGTTA...,GGTGTGTTGCACAGT,MK:tile_14356|chr15-67031618+67031887|scramble...,15;270M;NM:i:1;MD:Z:121T148;60
93061713,MK:tile_14356|chr15-67031618+67031887|scramble...,AGGACCGGATCAACTTGAAGCCCCTGATTCTGTTAGAATAAGGTTA...,TGTTGCGTAGACCGC,MK:tile_14356|chr15-67031618+67031887|scramble...,15;270M;NM:i:1;MD:Z:75T194;60
93061714,MK:tile_14356|chr15-67031618+67031887|scramble...,AGGACCGGATCAACTTGAAGCCCCTGATTCTGTTAGAATAAGGTTA...,GGTATGTAACAGATG,MK:tile_14356|chr15-67031618+67031887|scramble...,15;270M;NM:i:1;MD:Z:75T194;60


In [7]:
# take all rows with non NA value in BC and get the number of header names
assigned_sequences = assigned_sequences[assigned_sequences["bc"].notna()] # 93057876 


In [ ]:
assigned_sequences

In [8]:
new_bc_count_table = assigned_sequences.groupby("header").count() # 76375 => 3840 missing still 

In [ ]:
new_bc_count_table.shape[0]

In [9]:
# check for bc > threshold
new_bc_count_table[new_bc_count_table["bc"]>=5].shape[0]
# new_bc_count_table[new_bc_count_table["bc"]>=10].shape[0]

75719

In [10]:
new_bc_count_table[new_bc_count_table["bc"]>=10].shape[0]

75398